In [17]:
# Diagnosztika

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

url = "https://www.tippmixpro.hu/hu/elo/i/elo-esemenyek/100/league-of-legends-lol/vilag/lol-world-championship-2025/gen-g-esports-kt-rolster/285417910678360064/all"

chrome_options = Options()
# chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

try:
    print("Oldal betöltése...")
    driver.get(url)

    # Cookie elfogadás
    try:
        cookie_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        cookie_btn.click()
        time.sleep(1)
    except Exception:
        pass

    # Váltás az első iframe-re
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "iframe")))
    iframe = driver.find_elements(By.TAG_NAME, "iframe")[0]
    driver.switch_to.frame(iframe)
    time.sleep(2)

    # Article elemek begyűjtése
    articles = driver.find_elements(By.TAG_NAME, "article")
    print(f"{len(articles)} market található.")

    # Az első 2 article teljes HTML-je
    for i, art in enumerate(articles[:2]):
        html = art.get_attribute("outerHTML")
        print(f"\n--- ARTICLE {i+1} HTML snippet ---")
        print(html[:800].replace("\n", "") + " ...")

        with open(f"article_{i+1}.html", "w", encoding="utf-8") as f:
            f.write(html)

    print("\nKimentve: article_1.html, article_2.html")

except Exception as e:
    print("Hiba:", e)
finally:
    driver.quit()


Oldal betöltése...
22 market található.

--- ARTICLE 1 HTML snippet ---
<article class="Market Market--Id-466 Market--Part-1889 Market--Sport-100 Market--Order-0 Market--Template-4 Market--Column-2 Market--label-none"><div class="Market__Legend"><button type="button" class="Market__CollapseBtn"><span class="Market__CollapseIcon"><span class="OM-Icon OM-Icon--Svg OM-Icon--general OM-Icon--remove OM-Icon--Medium1"><svg viewBox="0 0 126 126"><path fill-rule="evenodd" clip-rule="evenodd" d="M63 59.9279L18.9279 104L-8.27363e-07 85.0721L63 22.0721L126 85.0721L107.072 104L63 59.9279Z"></path></svg></span></span><div class="Market__CollapseInfo"><span class="Market__CollapseText" title="Ki nyeri? - Teljes mérkőzés">Ki nyeri? - Teljes mérkőzés</span></div></button><button type="button" class="FavoriteMarketsButton"><span class="OM-Icon OM-Icon--Svg OM-Icon--general  ...

--- ARTICLE 2 HTML snippet ---
<article class="Market Market--Id-322 Market--Part-1889 Market--Sport-100 Market--Order-0 Mark

In [21]:
# Tippmix live iframe scrape

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

url = "https://www.tippmixpro.hu/hu/elo/i/elo-esemenyek/100/league-of-legends-lol/vilag/lol-world-championship-2025/gen-g-esports-kt-rolster/285417910678360064/palyak"

chrome_options = Options()
chrome_options.add_argument("--headless=new")  # ha nem akarsz böngészőt látni, vedd ki a kommentet
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

try:
    print("Oldal betöltése...")
    driver.get(url)

    # Cookie elfogadás
    try:
        cookie_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        cookie_btn.click()
        time.sleep(1)
    except Exception:
        pass

    # Váltás az első iframe-re
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "iframe")))
    iframe = driver.find_elements(By.TAG_NAME, "iframe")[0]
    driver.switch_to.frame(iframe)
    time.sleep(2)

    # Article elemek begyűjtése
    articles = driver.find_elements(By.TAG_NAME, "article")
    print(f"{len(articles)} market található.")

    for art in articles:
        try:
            # Market neve
            legend_el = art.find_element(By.CLASS_NAME, "Market__CollapseText")
            legend = legend_el.get_attribute("title") or legend_el.text.strip()
            print(f"\n=== {legend} ===")

            # Odds gombok keresése
            odds_buttons = art.find_elements(By.CLASS_NAME, "OddsButton")
            for btn in odds_buttons:
                try:
                    name_el = btn.find_element(By.CLASS_NAME, "OddsButton__Text")
                    odds_el = btn.find_element(By.CLASS_NAME, "OddsButton__Odds")
                    name = name_el.text.strip()
                    odds = odds_el.text.strip()
                    print(f"{name}: {odds}")
                except Exception:
                    # vannak olyan gombok, ahol nincs név (pl. over/under)
                    odds_el = btn.find_element(By.CLASS_NAME, "OddsButton__Odds")
                    odds = odds_el.text.strip()
                    print(f"Odds: {odds}")
        except Exception:
            continue

except Exception as e:
    print("Hiba:", e)
finally:
    driver.quit()


Oldal betöltése...
22 market található.

=== 1X2 - Rendes játékidő ===
Tunari: 4,30
Döntetlen: 3,00
CSM Resita: 1,83

=== Mindkét csapat szerez gólt - Rendes játékidő ===
Igen: 2,38
Nem: 1,45

=== 1X2 + Mindkét csapat szerez gólt - Rendes játékidő ===
Hazai és Nem: 5,75
Hazai és Igen: 11,50
Döntetlen és Igen: 5,50
Döntetlen és Nem: 6,50
Vendég és Nem: 2,67
Vendég és Igen: 6,00

=== Döntetlennél a tét visszajár - Rendes játékidő ===
Tunari: 2,80
CSM Resita: 1,32

=== Kétesély - Rendes játékidő ===
Tunari vagy Döntetlen: 1,73
Tunari vagy CSM Resita: 1,27
CSM Resita vagy Döntetlen: 1,16

=== Hendikep - Rendes játékidő ===
Tunari (-3): 100,00
Döntetlen - (Tunari -3): 35,00

=== Gólszám - Rendes játékidő ===
Odds: 1,07
Odds: 5,50
Odds: 1,49
Odds: 2,26
Odds: 1,64
Odds: 2,00
Odds: 1,93
Odds: 1,69
Odds: 2,22
Odds: 1,51
Odds: 2,53
Odds: 1,40
Odds: 4,90
Odds: 1,10
Odds: 9,25

=== Gólszám - 1. félidő ===

=== Gólszám - 2. félidő ===

=== Gólszám - Rendes játékidő ===

=== Gólszám - 1. félidő ===
